In [32]:
import os
import shutil

import pandas as pd
import numpy as np
import json

In [33]:
df=pd.DataFrame(columns=['jd','resume','macro_scores','micro_scores'])

In [34]:
df

,jd,resume,macro_scores,micro_scores


In [35]:
data_to_load=[x for x in os.listdir("resume-score-details") if (x.split(".")[-1]=="json" and x.split("_")[0]!="invalid")]
data_to_load

['empty_additional_info_0.json',
 'empty_additional_info_1.json',
 'empty_additional_info_10.json',
 'empty_additional_info_11.json',
 'empty_additional_info_110.json',
 'empty_additional_info_111.json',
 'empty_additional_info_112.json',
 'empty_additional_info_113.json',
 'empty_additional_info_114.json',
 'empty_additional_info_115.json',
 'empty_additional_info_116.json',
 'empty_additional_info_117.json',
 'empty_additional_info_118.json',
 'empty_additional_info_119.json',
 'empty_additional_info_12.json',
 'empty_additional_info_120.json',
 'empty_additional_info_121.json',
 'empty_additional_info_122.json',
 'empty_additional_info_123.json',
 'empty_additional_info_124.json',
 'empty_additional_info_125.json',
 'empty_additional_info_13.json',
 'empty_additional_info_14.json',
 'empty_additional_info_15.json',
 'empty_additional_info_16.json',
 'empty_additional_info_17.json',
 'empty_additional_info_18.json',
 'empty_additional_info_19.json',
 'empty_additional_info_2.json',
 

In [36]:

def fetch_json_data(data_to_load):
    df=pd.DataFrame()
    for d in data_to_load:
        with open(os.path.join("resume-score-details", d), "r") as file:
            data = json.load(file)

            row = {
                "job_description": data['input']['job_description'],
                "resume": data['input']['resume'],
                "macro_scores": data['output']['scores']['aggregated_scores']['macro_scores'],
                "micro_scores": data['output']['scores']['aggregated_scores']['micro_scores']
            }

            df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    return df        

In [37]:
df=fetch_json_data(data_to_load=data_to_load)
df.head()

,job_description,resume,macro_scores,micro_scores
0,## Job Title: Senior Electrical Engineering Pr...,"ABDULLAH JAWAID \nGulshan e Iqbal, block 2, Ka...",9.00,9.00
1,## Job Title: Senior Electrical Engineering Pr...,Team Lead \n Experience Required: 10 to 15 ...,7.00,6.00
2,## Job Title: Senior Electrical Project Manage...,Team Lead \n Experience Required: 10 to 15 ...,9.00,8.09
3,# Job Title: Electrical Engineer - Automation ...,MUHAMMAD NADEEM\nELECTRICAL ENGINEER\nPROFILE...,5.96,7.28
4,### Job Title: International Business Developm...,"Milat Road DHA 3, Lahore https://www.linkedin....",3.42,5.00


In [38]:
df.shape

(889, 4)

In [39]:
df.to_csv("final_data.csv",index=False,escapechar="\\")

In [40]:
import nltk
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import re

STOPWORDS = set(list(ENGLISH_STOP_WORDS)) - {"not", "no", "nor"}

def preprocess_text(text):
    if not isinstance(text, str):
        return ""

    text = re.sub(r'http\S+|www\S+|https\S+', ' ', text)

    text = re.sub(r'\S+@\S+', ' ', text)

    text = re.sub(r'<[^>]+>', ' ', text)

    text = re.sub(r'\s+', ' ', text).strip()

    text = re.sub(r"(\'re)", " are", text)
    text = re.sub(r"(\'s)", " is", text)
    text = re.sub(r"(\'ve)", " have", text)
    text = re.sub(r"(n\'t)", " not", text)
    text = re.sub(r"(\'ll)", " will", text)
    text = re.sub(r"(\'d)", " would", text)
    text = re.sub(r"(\'m)", " am", text)

    text = text.lower()

    text = re.sub(r'[^a-z\s]', ' ', text)

    tokens = [tok for tok in text.split() if len(tok) > 2 and tok not in STOPWORDS]

    return " ".join(tokens)

In [41]:
df['resume']=df['resume'].apply(preprocess_text)
df['job_description']=df['job_description'].apply(preprocess_text)

In [42]:
df

,job_description,resume,macro_scores,micro_scores
0,job title senior electrical engineering projec...,abdullah jawaid gulshan iqbal block karachi su...,9.00,9.00
1,job title senior electrical engineering projec...,team lead experience required years relevant e...,7.00,6.00
2,job title senior electrical project manager co...,team lead experience required years relevant e...,9.00,8.09
3,job title electrical engineer automation contr...,muhammad nadeem electrical engineer profile mo...,5.96,7.28
4,job title international business development m...,milat road dha lahore scan data entry receivin...,3.42,5.00
...,...,...,...,...
884,digital marketing strategist innovatetech corp...,sohaib afzal lahore pakistan professional summ...,7.00,4.32
885,join apex global solutions business developmen...,hassan javed artificial intelligence developer...,5.00,5.10
886,business development manager join fictional co...,umerqayyum profile tofindachallengingjobinadyn...,3.30,2.22
887,join apex global solutions business developmen...,sharoon bakhshi phone email lums edu lahore pa...,4.87,4.45


In [43]:
def prepare_input(sample):
    resume = sample['resume']
    jd = sample['job_description']
    
    text = resume + " [SEP] " + jd
    
    macro = sample["macro_scores"]
    micro = sample["micro_scores"]
    
    return text, [macro, micro]

In [44]:
from transformers import AutoTokenizer

tokenizer=AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize(text):
    return tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )

c:\Users\vansh\Projects\ML-Learner\python_backend\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [48]:
tokenize("Hello world")

{'input_ids': tensor([[ 101, 7592, 2088,  102,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,

In [52]:
import torch
from torch.utils.data import DataLoader,Dataset

class ResumeDataset(Dataset):
    def __init__(self,data):
        self.sample=[prepare_input(x) for _,x in data.iterrows()]

    def __len__(self):
        return len(self.sample)


    def __getitem__(self, idx):    
        text,labels=self.sample[idx]

        enc=tokenize(text)

        return {
            "input_idx":enc['input_ids'].squeeze(0),
            "attention_mask":enc['attention_mask'].squeeze(0),
            "labels":torch.tensor(labels,dtype=torch.float)

        }

In [53]:
import torch.nn as nn
from transformers import AutoModel

class ResumeScore(nn.Module):
    def __init__(self, ):
        super().__init__()

        self.bert=AutoModel.from_pretrained("bert-base-uncased")

        self.regressor=nn.Sequential(
            nn.Linear(768,256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256,2) # macro ,micro

        )

    def forward(self,input_ids,attention_mask):
        outputs=self.bert(input_ids,attention_mask)

        cls_output=outputs.last_hidden_state[:,0] # cls token

        return self.regressor(cls_output)
        

In [ ]:
dataset=ResumeDataset(data=df)

dataset_loader=DataLoader(dataset=dataset,batch_size=8,shuffle=True)

model=ResumeScore()
optimizer=torch.optim.AdamW(model.parameters(),lr=2e-5)

loss_fn=nn.MSELoss()



In [ ]:
epochs=10
from tqdm import tqdm
for epoch in tqdm(range(epochs)):
    for batch in tqdm(loader):
        optimizer.zero_grad()
        output=model(batch["input_ids"], batch["attention_mask"])

        loss=loss_fn(output,batch['labels'])

        loss.backward()

        optimizer.step()

    print("Epoch done",loss.item())    
